# CGR Bridge Protocol: Colab verification

This notebook tests the runtime handoff only. It uses the coder LoRA on the exact checkpoint named by its adapter config: `Qwen/Qwen3-0.6B`. It does not train adapters, download datasets, or use Kaggle. Five synthetic chains are embedded below.

For each chain it will: prefill a base-model cache containing verified steps plus a deliberately bad tail; crop that cache to the verified prefix and assert the retained K/V tensors are bit-identical; generate Steps 6–8 with the coder adapter; prefill the generated patch with the base policy and no generation; compare Step 9 logits against a full base prefill; then greedily generate Step 9 text and record latency, max absolute logit delta, logit scale, and argmax agreement.

This is a systems check, not a code-correctness benchmark. Read the generated patches and Step 9 outputs. Run on a Colab GPU. Float32 is used for this verification run so logits are easier to compare; the measured latency is not a deployment-speed claim.

**Exact checkpoint rule:** this test uses `Qwen/Qwen3-0.6B` because that is the coder adapter's declared base. It intentionally does not load the adapter onto `Qwen/Qwen3-0.6B-Base`. The cyber adapter is excluded because it declares a different `unsloth/Qwen3-0.6B` checkpoint.


In [ ]:
# Pin the cache API used in this notebook; 4.57.6 uses crop(max_length) with an absolute token length.
%pip -q install transformers==4.57.6 "peft>=0.15,<1" accelerate safetensors huggingface_hub


In [ ]:
import copy, json, os, time
from datetime import datetime, timezone
from pathlib import Path
import torch
import transformers
from huggingface_hub import hf_hub_download, snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from google.colab import drive

# Drive stores per-case progress so completed measurements survive a Colab disconnect.
drive.mount('/content/drive')
RUN_ID = 'bridge_qwen3_0.6b_coder_synth5_v1'  # change this for an independent run
RUN_DIR = Path('/content/drive/MyDrive/cgr_bridge_verification') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = RUN_DIR / 'progress.jsonl'
RESUME_PARTIAL_RUN = True
os.environ['HF_HOME'] = '/content/drive/MyDrive/cgr_hf_cache'  # persist base/model downloads across Colab sessions

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab (Runtime → Change runtime type → T4 GPU).")
DEVICE = torch.device('cuda:0')
torch.manual_seed(41)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name(0))
print('Transformers:', transformers.__version__)


## 1. Load the coder adapter on its declared base checkpoint

The adapter card identifies `Qwen/Qwen3-0.6B` as its parent. This notebook enforces an exact model ID match before loading. The source adapter targets q/k/v/o, so adapter-generated cache entries are not reused by the base continuation; only the accepted patch text is re-prefilled under the base policy.


In [ ]:
BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
ADAPTER_ID = 'ViswanthSai/qwen0.6B-ViswanthSai-lora'

adapter_cfg_path = hf_hub_download(ADAPTER_ID, 'adapter_config.json')
with open(adapter_cfg_path, encoding='utf-8') as f:
    adapter_cfg = json.load(f)
ADAPTER_BASE_ID = (adapter_cfg.get('base_model_name_or_path') or '').rstrip('/')
if ADAPTER_BASE_ID != BASE_MODEL_ID:
    raise ValueError(f'Exact base mismatch: adapter={ADAPTER_BASE_ID!r}, selected={BASE_MODEL_ID!r}')
print('Verified exact base:', ADAPTER_BASE_ID)
print('LoRA target modules:', adapter_cfg.get('target_modules'))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float32, device_map={'': 0}, low_cpu_mem_usage=True
)
ADAPTER_PATH = snapshot_download(ADAPTER_ID, local_dir=RUN_DIR / 'pretrained_coder_adapter')
model = PeftModel.from_pretrained(base, ADAPTER_PATH, adapter_name='coder', is_trainable=False)
model.eval()
model.config.use_cache = True
model.set_adapter('coder')
print('Base parameters:', sum(p.numel() for p in base.parameters()))
print('Adapter parameters:', sum(p.numel() for name, p in model.named_parameters() if 'lora_' in name))


## 2. Five inline synthetic chains

Each chain has five verified steps and a deliberately corrupted tail. The patch request asks the adapter to replace only Steps 6–8. These examples are fixed strings in the notebook; no external data source is used.


In [ ]:
SYNTHETIC_CHAINS = [
    {
        'name':'csv_totals',
        'task':'Implement a Python function that returns total amounts by category from CSV-like row dictionaries.',
        'verified': [
            'Step 1: The input is a list of dictionaries with category and amount fields.',
            'Step 2: Ignore rows whose amount cannot be converted to float.',
            'Step 3: Preserve first-seen category order.',
            'Step 4: Accumulate each valid amount under its category.',
            'Step 5: Return a dictionary from category to total.',
        ],
        'bad_tail':['Step 6: totals = []', 'Step 7: totals.append(row)', 'Step 8: return totals'],
        'repair':'Replace the bad tail with concise Python implementation steps for the stated function.'
    },
    {
        'name':'normalize_names',
        'task':'Implement a Python function that normalizes a list of names and removes duplicates while keeping order.',
        'verified': [
            'Step 1: Accept a list of strings that may contain extra whitespace.',
            'Step 2: Strip leading and trailing whitespace.',
            'Step 3: Normalize comparison using case-folding.',
            'Step 4: Keep the first spelling of each normalized name.',
            'Step 5: Return the retained names in their original order.',
        ],
        'bad_tail':['Step 6: names.sort()', 'Step 7: names = set(names)', 'Step 8: return list(names)'],
        'repair':'Replace the bad tail with concise Python implementation steps for the stated function.'
    },
    {
        'name':'inventory_restock',
        'task':'Implement a Python function that reports inventory items below their restock threshold.',
        'verified': [
            'Step 1: The input maps item names to integer quantities.',
            'Step 2: A second mapping gives each item its minimum quantity.',
            'Step 3: Missing threshold entries use zero.',
            'Step 4: Include an item only when its quantity is strictly below threshold.',
            'Step 5: Return the selected item names sorted alphabetically.',
        ],
        'bad_tail':['Step 6: low = [q for q in inventory]', 'Step 7: low.sort(reverse=True)', 'Step 8: return low'],
        'repair':'Replace the bad tail with concise Python implementation steps for the stated function.'
    },
    {
        'name':'event_dedup',
        'task':'Implement a Python function that keeps the newest event for each event_id.',
        'verified': [
            'Step 1: Each event has an event_id and an integer timestamp.',
            'Step 2: The input may contain multiple events with the same id.',
            'Step 3: Keep the event with the greatest timestamp for each id.',
            'Step 4: If timestamps tie, keep the event that appears later in the input.',
            'Step 5: Return the selected events sorted by event_id.',
        ],
        'bad_tail':['Step 6: selected = set(events)', 'Step 7: selected.sort()', 'Step 8: return selected'],
        'repair':'Replace the bad tail with concise Python implementation steps for the stated function.'
    },
    {
        'name':'safe_lookup',
        'task':'Implement a Python function that reads a nested mapping by a sequence of keys and returns a default when a key is absent.',
        'verified': [
            'Step 1: Accept a mapping, a list of keys, and a default value.',
            'Step 2: Visit keys from left to right.',
            'Step 3: Continue only while the current value is a mapping containing the key.',
            'Step 4: Return the default immediately when a key cannot be followed.',
            'Step 5: Return the final nested value when every key is present.',
        ],
        'bad_tail':['Step 6: value = mapping[keys]', 'Step 7: if not value: return None', 'Step 8: return value'],
        'repair':'Replace the bad tail with concise Python implementation steps for the stated function.'
    },
]
assert len(SYNTHETIC_CHAINS) == 5


## 3. Cache and timing helpers

`DynamicCache.crop(prefix_len)` uses an absolute target token length in the pinned Transformers version. The bit-identity assertion compares every retained key/value tensor with the same prefix slice from before cropping. Logit equivalence is tested separately after bridge prefill.


In [ ]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def timed(fn):
    sync()
    start = time.perf_counter()
    value = fn()
    sync()
    return value, (time.perf_counter() - start) * 1000.0

def ids_for(text):
    return tokenizer(text, add_special_tokens=False, return_tensors='pt').input_ids.to(DEVICE)

def base_forward(input_ids, past_key_values=None, attention_mask=None):
    if attention_mask is None:
        past_len = past_key_values.get_seq_length() if past_key_values is not None else 0
        attention_mask = torch.ones((1, past_len + input_ids.shape[1]), dtype=torch.long, device=DEVICE)
    with model.disable_adapter(), torch.inference_mode():
        return model(input_ids=input_ids, past_key_values=past_key_values,
                     attention_mask=attention_mask, use_cache=True, return_dict=True)

def cache_kv_snapshot(cache):
    return [(layer.keys.detach().clone(), layer.values.detach().clone()) for layer in cache.layers]

def crop_is_bit_identical(before, cache, prefix_len):
    if cache.get_seq_length() != prefix_len or len(before) != len(cache.layers):
        return False
    for (old_k, old_v), layer in zip(before, cache.layers):
        if not torch.equal(layer.keys, old_k[..., :prefix_len, :]):
            return False
        if not torch.equal(layer.values, old_v[..., :prefix_len, :]):
            return False
    return True

def continue_base_greedy(first_logits, past_key_values, max_new_tokens=48):
    # first_logits predicts the first Step 9 token after the base-only patch prefill.
    next_id = first_logits.argmax(dim=-1, keepdim=True)
    generated = []
    past = past_key_values
    eos = tokenizer.eos_token_id
    for _ in range(max_new_tokens):
        token_id = int(next_id.item())
        if eos is not None and token_id == eos:
            break
        generated.append(token_id)
        past_len = past.get_seq_length()
        mask = torch.ones((1, past_len + 1), dtype=torch.long, device=DEVICE)
        with model.disable_adapter(), torch.inference_mode():
            out = model(input_ids=next_id, past_key_values=past, attention_mask=mask,
                        use_cache=True, return_dict=True)
        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    return tokenizer.decode(generated, skip_special_tokens=True)


## 4. Run crop → adapter patch → base bridge → Step 9

For each chain the full bad-tail context is prefetched under the base, then cropped back to the token count of Steps 1–5. The adapter receives a repair directive after that cached prefix. Its generated text is decoded and retokenized as the patch. The bridge feeds those patch tokens through the base model with adapters disabled and `use_cache=True`; it does not call `generate`.

The reference logits come from a fresh base prefill over the exact same token sequence (verified prefix IDs + patch IDs). We report `max |Δlogit|`, absolute logit scale, argmax agreement, and separate crop, adapter generation, bridge prefill, and Step 9 continuation latency.


In [ ]:
def run_chain(case):
    prefix_text = (
        f"Task: {case['task']}\n"
        "Continue as numbered steps. Keep the implementation consistent with the task and prior steps.\n"
        + '\n'.join(case['verified']) + '\n'
    )
    bad_tail_text = '\n'.join(case['bad_tail'])
    prefix_ids = ids_for(prefix_text)
    bad_tail_ids = ids_for(bad_tail_text)
    full_bad_ids = torch.cat([prefix_ids, bad_tail_ids], dim=1)
    prefix_len = prefix_ids.shape[1]

    # Base prefill with the deliberately corrupted tail, then crop at the verified boundary.
    full_mask = torch.ones_like(full_bad_ids)
    full_out, prefill_bad_ms = timed(lambda: base_forward(full_bad_ids, attention_mask=full_mask))
    cache = full_out.past_key_values
    if not isinstance(cache, DynamicCache):
        raise TypeError(f'Expected DynamicCache, got {type(cache)}')
    before = cache_kv_snapshot(cache)
    crop_out, crop_ms = timed(lambda: cache.crop(prefix_len))
    crop_pass = crop_is_bit_identical(before, cache, prefix_len)
    if not crop_pass:
        raise AssertionError(f"DynamicCache.crop changed retained prefix KV for {case['name']}")

    # Keep independent copies because generation mutates cache state in place.
    adapter_cache = copy.deepcopy(cache)
    bridge_cache = copy.deepcopy(cache)
    repair_messages = [{
        'role':'user',
        'content': (
            f"{case['repair']}\n"
            f"Task: {case['task']}\n"
            "Output Steps 6, 7, and 8 only. Keep them concise and continue from the verified Steps 1–5."
        ),
    }]
    repair_prompt = tokenizer.apply_chat_template(
        repair_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    repair_ids = ids_for(repair_prompt)
    # Pass the complete token sequence so generation can separate the cached prefix from new repair-prompt tokens.
    generation_input_ids = torch.cat([prefix_ids, repair_ids], dim=1)
    adapter_mask = torch.ones_like(generation_input_ids)
    model.set_adapter('coder')
    gen_out, adapter_ms = timed(lambda: model.generate(
        input_ids=generation_input_ids, past_key_values=adapter_cache, attention_mask=adapter_mask,
        max_new_tokens=160, do_sample=False, use_cache=True,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    ))
    patch_text = tokenizer.decode(gen_out[0, generation_input_ids.shape[1]:], skip_special_tokens=True).strip()
    if not patch_text:
        raise AssertionError(f"Adapter generated an empty patch for {case['name']}")
    patch_ids = ids_for('\n' + patch_text)

    # Bridge: base-only, zero-generation prefill over patch tokens.
    bridge_mask = torch.ones((1, prefix_len + patch_ids.shape[1]), dtype=torch.long, device=DEVICE)
    bridge_out, bridge_ms = timed(lambda: base_forward(
        patch_ids, past_key_values=bridge_cache, attention_mask=bridge_mask
    ))

    # Control: full base prefill over precisely the same token IDs.
    reference_ids = torch.cat([prefix_ids, patch_ids], dim=1)
    reference_mask = torch.ones_like(reference_ids)
    reference_out, reference_ms = timed(lambda: base_forward(
        reference_ids, attention_mask=reference_mask
    ))
    bridged_logits = bridge_out.logits[:, -1, :].float()
    reference_logits = reference_out.logits[:, -1, :].float()
    max_delta = float((bridged_logits - reference_logits).abs().max().item())
    max_abs_bridge = float(bridged_logits.abs().max().item())
    max_abs_reference = float(reference_logits.abs().max().item())
    scale_ratio = max_abs_bridge / max(max_abs_reference, 1e-12)
    argmax_same = bool(torch.equal(bridged_logits.argmax(-1), reference_logits.argmax(-1)))
    finite = bool(torch.isfinite(bridged_logits).all() and torch.isfinite(reference_logits).all())

    step9_text, step9_ms = timed(lambda: continue_base_greedy(
        bridge_out.logits[:, -1, :], bridge_out.past_key_values, max_new_tokens=48
    ))
    result = {
        'case':case['name'], 'crop_bit_identical':crop_pass,
        'prefix_tokens':int(prefix_len), 'bad_tail_tokens':int(bad_tail_ids.shape[1]),
        'base_prefill_bad_context_ms':prefill_bad_ms,
        'patch_tokens':int(patch_ids.shape[1]), 'crop_ms':crop_ms,
        'adapter_steps_6_8_ms':adapter_ms, 'bridge_prefill_ms':bridge_ms,
        'full_base_reference_prefill_ms':reference_ms, 'step9_generation_ms':step9_ms,
        'max_abs_logit_delta':max_delta, 'max_abs_bridged_logit':max_abs_bridge,
        'max_abs_reference_logit':max_abs_reference, 'logit_scale_ratio':scale_ratio,
        'step9_argmax_matches_full_base':argmax_same, 'logits_finite':finite,
        'step9_text':step9_text, 'adapter_patch':patch_text,
    }
    return result

if not RESUME_PARTIAL_RUN and PROGRESS_PATH.exists():
    PROGRESS_PATH.unlink()

completed_rows = {}
if RESUME_PARTIAL_RUN and PROGRESS_PATH.exists():
    for line in PROGRESS_PATH.read_text(encoding='utf-8').splitlines():
        try:
            record=json.loads(line)
            if record.get('status') == 'complete':
                completed_rows[record['case']] = record
        except (json.JSONDecodeError, KeyError):
            continue  # ignore an incomplete final line after an abrupt runtime stop

results=list(completed_rows.values())
for case in SYNTHETIC_CHAINS:
    if case['name'] in completed_rows:
        print(f"Skipping already completed case: {case['name']}")
        continue
    print(f"Starting case: {case['name']}")
    try:
        row=run_chain(case)
        row['status']='complete'
        row['completed_utc']=datetime.now(timezone.utc).isoformat()
    except Exception as exc:
        row={'case':case['name'],'status':'error','error':repr(exc),
             'logged_utc':datetime.now(timezone.utc).isoformat()}
        with PROGRESS_PATH.open('a',encoding='utf-8') as f:
            f.write(json.dumps(row,ensure_ascii=False)+'\n')
            f.flush(); os.fsync(f.fileno())
        raise
    with PROGRESS_PATH.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
        f.flush(); os.fsync(f.fileno())
    results.append(row)
    completed_rows[case['name']]=row
    print(f"\n[{row['case']}] crop exact={row['crop_bit_identical']} | Δmax={row['max_abs_logit_delta']:.3e} | scale={row['logit_scale_ratio']:.3f} | Step9 argmax match={row['step9_argmax_matches_full_base']}")
    print(f"latency ms: crop {row['crop_ms']:.2f}, adapter 6–8 {row['adapter_steps_6_8_ms']:.2f}, bridge prefill {row['bridge_prefill_ms']:.2f}, Step9 {row['step9_generation_ms']:.2f}")
    print('Adapter patch:\n', row['adapter_patch'])
    print('Base Step 9 continuation:\n', row['step9_text'])


## 5. Summarize and save measurements

Interpretation: `crop_bit_identical` checks exact K/V preservation for the retained prefix. `max_abs_logit_delta` compares the next-token base logits after bridge prefill with a fresh full-base prefill over the same token IDs. `step9_argmax_matches_full_base`, finite logits, and the scale ratio are coherence checks. Inspect the generated Step 9 text for readability; this test does not establish semantic correctness of the generated patch.


In [ ]:
LOGIT_TOL = 1e-4
SCALE_RATIO_LIMIT = 2.0
summary = {
    'base_model_id':BASE_MODEL_ID, 'adapter_id':ADAPTER_ID,
    'transformers_version':transformers.__version__, 'dtype':'float32',
    'synthetic_case_count':len(results), 'max_abs_logit_delta_all':max(r['max_abs_logit_delta'] for r in results),
    'all_crop_bit_identical':all(r['crop_bit_identical'] for r in results),
    'all_logits_finite':all(r['logits_finite'] for r in results),
    'all_scale_ratios_below_limit':all(r['logit_scale_ratio'] < SCALE_RATIO_LIMIT for r in results),
    'all_argmax_match':all(r['step9_argmax_matches_full_base'] for r in results),
    'tolerance_max_abs_logit_delta':LOGIT_TOL,
    'all_within_logit_tolerance':all(r['max_abs_logit_delta'] <= LOGIT_TOL for r in results),
    'results':results,
}
print(json.dumps({k:v for k,v in summary.items() if k!='results'}, indent=2))

out_path=RUN_DIR/'bridge_verification_results.json'
out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:',out_path)


## Scope and reading the result

This run tests a specific exact-base adapter pair on five synthetic prompts. It does not test the cyber adapter, a different base checkpoint, DAG localization, semantic repair success, or deployment latency on the research machine. A pass requires exact retained-prefix K/V slices, finite base logits, logit-scale ratio below 2, argmax agreement, and `max |Δlogit| ≤ 1e-4` on all five cases. If a threshold fails, inspect the saved per-case values and generated text; do not relax the threshold just to obtain a pass.

Transformers 4.57.6 cache source documents `DynamicCache.crop(max_length)` as an absolute token-length crop and stores per-layer K/V in `cache.layers`; see the [pinned cache implementation](https://github.com/huggingface/transformers/blob/v4.57.6/src/transformers/cache_utils.py) and [cache docs](https://huggingface.co/docs/transformers/kv_cache).
